# ✈️ Extracción y Procesamiento Inicial de Datos de Tráfico Aéreo (OpenSky)

La disponibilidad de datos aeronáuticos en tiempo real permite analizar patrones de movilidad aérea, comportamiento de aeronaves y variaciones operativas a escala global.  
Este proyecto desarrolla un **pipeline ETL** que ingesta información del endpoint público de **OpenSky Network**, centrado en capturar el estado actual de miles de vuelos activos en simultáneo.

La API proporciona variables clave como:

- identificador **ICAO24**  
- país de origen  
- latitud / longitud  
- altitudes barométrica y geométrica  
- velocidad, rumbo, tasa vertical  
- estado en tierra o en vuelo  
- timestamp del servidor  

Estos datos se transforman y almacenan en un **Data Lake local** siguiendo la arquitectura **Bronze → Silver → Gold**, lo que permite realizar análisis históricos, construir métricas aeronáuticas y preparar la futura migración a entornos de nube (Azure).

## Objetivos

**Extracción (Bronze):**
- Consumir el endpoint `states/all` de OpenSky.  
- Normalizar la estructura JSON y convertirla en tabla.  
- Incorporar timestamps (servidor y extracción).  
- Guardar los datos crudos en **Delta Lake**.

**Transformación (Silver):**
- Limpiar valores faltantes y tipos de datos.  
- Estandarizar columnas y coordenadas.  
- Preparar la tabla para análisis temporal.

**Métricas (Gold):**
- Crear features de movilidad aérea: altitud efectiva, variación de velocidad, indicadores de vuelo/estacionamiento, etc.  
- Generar datasets optimizados para visualización y análisis exploratorio.

## Alcance y supuestos

- Se utilizan exclusivamente datos públicos provistos por OpenSky Network.  
- La extracción se realiza bajo límites de la API pública (sin autenticación obligatoria).  
- El objetivo es **práctico y educativo**, orientado al portfolio de Ingeniería de Datos.

## Reproducibilidad

- Dependencias detalladas en `requirements.txt`.  
- Las rutas del Data Lake se configuran en `pipeline.conf`.  
- Todas las funciones auxiliares se encuentran en `src/etl_utils.py`.

---

**Estructura del notebook:**

0) Configuración inicial  
1) Extracción del endpoint `states/all`  
2) Normalización del JSON  
3) Limpieza y estandarización mínima  
4) Almacenamiento en Delta Lake (capa Bronze)  
5) Verificación y vista preliminar de los datos  

## 0. Configuración inicial

En este paso se importan todas las librerías necesarias y las funciones auxiliares definidas en `etl_utils.py`.  
Este enfoque permite mantener el notebook **ordenado, modular y fácilmente reproducible**, centralizando en un único módulo las operaciones comunes del pipeline ETL: extracción desde la API pública de **OpenSky Network**, normalización del JSON, estandarización de columnas y escritura en las distintas capas del **Data Lake local** (Bronze → Silver → Gold).

El objetivo de esta sección es garantizar que todas las dependencias estén correctamente cargadas antes de iniciar el proceso de extracción y almacenamiento.

In [1]:
import sys
import os

# Se agrega la carpeta src al path (sube un nivel desde /notebooks)
sys.path.append(os.path.abspath("../src"))

# Importar funciones auxiliares
from etl_utils import *

# Librerías comunes
import pandas as pd

print("✅ Librerías importadas correctamente.")

✅ Librerías importadas correctamente.


## 1. Autenticación y lectura de configuración

La configuración del proyecto se administra mediante el archivo `pipeline.conf`,  
que centraliza parámetros como:
- la **URL base** de la API de OpenSky Network  
- credenciales opcionales para *Basic Auth* (en caso de usarse)  
- rutas del **Data Lake local**

Aunque la API pública de OpenSky no requiere autenticación obligatoria,almacenar parámetros en un archivo de configuración permite:
- mantener el notebook limpio  
- evitar credenciales expuestas en el código  
- facilitar la migración futura a servicios en la nube (Azure Key Vault)

El archivo se lee mediante `ConfigParser`, lo que permite obtener los valores  
en forma segura y reusable.


In [5]:
# Se instancia el parser y se lee el archivo de configuración
from configparser import ConfigParser

parser = ConfigParser()
parser.read("../pipeline.conf")

['../pipeline.conf']

In [6]:
# Parámetros de conexión
api_config = parser["api-opensky"]
base_url = api_config["base_url"]

In [7]:
print("📄 Configuración cargada correctamente.")
print(f"URL base: {base_url}")

📄 Configuración cargada correctamente.
URL base: https://opensky-network.org/api/states/all


In [11]:
# Prueba de conexión a la API OpenSky
response = requests.get(base_url)

if response.status_code == 200:
    data = response.json()
    print(f"La petición fue exitosa. Tipo de respuesta: {type(data)}")

    # Claves principales del JSON
    print(f"Claves principales recibidas: {list(data.keys())[:5]}")

    # Inspección parcial de 'states'
    print("\nPrimeras 2 aeronaves registradas:")
    pprint(data["states"][:2])

else:
    print(f"❌ Error en la petición: {response.status_code}, {response.content}")
print("✅ Prueba de conexión a la API realizada.")

La petición fue exitosa. Tipo de respuesta: <class 'dict'>
Claves principales recibidas: ['time', 'states']

Primeras 2 aeronaves registradas:
[['ab1644',
  'UAL2064 ',
  'United States',
  1763070488,
  1763070488,
  -110.5849,
  36.7424,
  10995.66,
  False,
  201.44,
  229.56,
  0,
  None,
  11437.62,
  None,
  False,
  0],
 ['aa3cbe',
  'N759PA  ',
  'United States',
  1763070433,
  1763070433,
  -111.834,
  41.801,
  1592.58,
  False,
  46.92,
  15.26,
  1.95,
  None,
  1638.3,
  None,
  False,
  0]]
✅ Prueba de conexión a la API realizada.
